# AdSafe Studio — risk detector, run 2

Fine-tunes YOLOv8 on the 11-class Open Images subset, on a free Colab GPU.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. The first cell
refuses to continue without one, because this run takes ~20 minutes on a T4 and
roughly 14 hours on Colab's CPU.

**What run 1 measured, and what this is trying to beat** (`training/RESULTS.md`):

| | run 1 | run 2 target |
|---|---|---|
| epochs | 12 | 40 |
| image size | 512 | 640 |
| classes | 10 | 11 |
| mAP@50 | 0.566 | — |

Run 1's biggest problem was not volume. `Wine` had the most training data of any
class and scored worst (0.250) because Open Images applies the label to bottles,
filled glasses and the liquid alike. It has been dropped; `Kitchen knife` and
`Bow and arrow` are new.

In [ ]:
# 1. Check the GPU before anything else.
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime -> Change runtime type -> T4 GPU, then run this again.\n"
        "On CPU this run takes roughly 14 hours instead of 20 minutes."
    )
print(torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=memory.total --format=csv,noheader

In [ ]:
# 2. Dependencies. albumentations matters here, not just ultralytics:
#    when it is present ultralytics adds blur, median-blur, grayscale and CLAHE
#    augmentation. Run 1 was trained on clean Open Images photographs and then
#    failed on compressed video of a motion-blurred weapon, so degradation the
#    model has never seen is exactly the gap worth closing.
!pip install -q ultralytics albumentations

import ultralytics
ultralytics.checks()

In [ ]:
# 3. Upload risk_dataset_colab.zip (~122 MB), produced by:
#       python training/package_for_colab.py
#
# Browser upload is fine at this size. For a slow connection, put the zip in
# Drive instead and swap this cell for:
#       from google.colab import drive; drive.mount('/content/drive')
#       !cp '/content/drive/MyDrive/risk_dataset_colab.zip' /content/
from google.colab import files

uploaded = files.upload()
print(uploaded.keys())

In [ ]:
# 4. Unpack and verify. A silently empty dataset trains happily and produces a
#    model that detects nothing, so the counts are checked before training.
import shutil, zipfile
from collections import Counter
from pathlib import Path

import yaml

DATA = Path('/content/risk')
if DATA.exists():
    shutil.rmtree(DATA)
with zipfile.ZipFile('/content/risk_dataset_colab.zip') as z:
    z.extractall(DATA)

spec = yaml.safe_load((DATA / 'dataset.yaml').read_text())
spec['path'] = str(DATA)          # absolute, so ultralytics resolves it from anywhere
(DATA / 'dataset.yaml').write_text(yaml.safe_dump(spec, sort_keys=False))
names = [spec['names'][i] for i in sorted(spec['names'])]

print(f"{len(names)} classes: {', '.join(names)}\n")
print(f"{'class':<16}{'train':>8}{'val':>8}")
print('-' * 32)
counts = {}
for split in ('train', 'val'):
    c = Counter()
    for f in (DATA / 'labels' / split).glob('*.txt'):
        for line in f.read_text().splitlines():
            if line.strip():
                c[int(line.split()[0])] += 1
    counts[split] = c
for i, n in enumerate(names):
    print(f'{n:<16}{counts["train"][i]:>8}{counts["val"][i]:>8}')

imgs = {s: len(list((DATA / 'images' / s).glob('*.jpg'))) for s in ('train', 'val')}
print('-' * 32)
print(f'{"IMAGES":<16}{imgs["train"]:>8}{imgs["val"]:>8}')
assert imgs['train'] > 100 and imgs['val'] > 50, 'dataset did not unpack correctly'
assert not [n for i, n in enumerate(names) if counts['val'][i] == 0], (
    'a class with no validation instances cannot be scored at all')

In [ ]:
# 5. Train. ~20 min on a T4 for yolov8n.
#
# MODEL: 'yolov8n.pt' keeps inference cheap, which matters because the app runs
# detection on a laptop CPU at two scales per frame. 'yolov8s.pt' scores better
# but is roughly 3x slower per frame -- on a 25 s clip that is the difference
# between about a minute and about three. Start with n.
from ultralytics import YOLO

MODEL = 'yolov8n.pt'
EPOCHS = 40
IMGSZ = 640

model = YOLO(MODEL)
results = model.train(
    data=str(DATA / 'dataset.yaml'),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=32,               # a T4 holds this comfortably at 640 px
    device=0,
    project='/content/runs',
    name='risk2',
    exist_ok=False,         # never overwrite a previous run's artefacts
    patience=12,            # stop early if validation stops improving
    # Scale jitter well above the 0.5 default. The failure this addresses is
    # measured: a pistol at arm's length in 720p video went undetected while the
    # same weapon filling the frame was found, because the model only ever saw
    # Open Images objects at photographic scale.
    scale=0.75,
    mosaic=1.0,             # composites four images; helps small objects
    close_mosaic=10,        # ...but disable it for the last 10 epochs
    fliplr=0.5,
    degrees=5.0,
)

In [ ]:
# 6. Per-class results, next to run 1. Read the per-class table, not just the
#    headline: run 1's average was dragged down by one badly-defined class.
RUN1 = {'Cocktail': 0.856, 'Knife': 0.766, 'Rifle': 0.749, 'Beer': 0.723,
        'Wine glass': 0.586, 'Handgun': 0.487, 'Dagger': 0.484, 'Sword': 0.420,
        'Shotgun': 0.339, 'Wine': 0.250}

metrics = model.val(data=str(DATA / 'dataset.yaml'), imgsz=IMGSZ, device=0)
print(f"\noverall mAP@50    {metrics.box.map50:.3f}   (run 1: 0.566)")
print(f"overall mAP@50-95 {metrics.box.map:.3f}   (run 1: 0.432)\n")
print(f"{'class':<16}{'mAP@50':>9}{'run 1':>9}{'change':>10}")
print('-' * 44)
for i, name in enumerate(names):
    ap = float(metrics.box.maps[i])
    was = RUN1.get(name)
    if was is None:
        print(f'{name:<16}{ap:>9.3f}{"new":>9}{"":>10}')
    else:
        print(f'{name:<16}{ap:>9.3f}{was:>9.3f}{ap - was:>+10.3f}')

In [ ]:
# 7. Download the weights and the evidence.
#
# Put risk_yolov8.pt in the project's models/ folder. train.py keeps the previous
# file as risk_yolov8.prev.pt when it writes there -- copying by hand does not, so
# rename the old one yourself before overwriting. A run that scores worse is a
# real possibility and you want the way back.
import shutil

from google.colab import files

run = Path('/content/runs/risk2')
shutil.copy(run / 'weights' / 'best.pt', '/content/risk_yolov8.pt')
shutil.make_archive('/content/run2_artifacts', 'zip', run)

files.download('/content/risk_yolov8.pt')       # -> models/
files.download('/content/run2_artifacts.zip')   # -> training/runs/, cited by RESULTS.md

## After downloading

1. Rename the existing `models/risk_yolov8.pt` to `models/risk_yolov8.prev.pt`.
2. Drop the new `risk_yolov8.pt` into `models/`.
3. Unzip `run2_artifacts.zip` into `training/runs/risk2/`.
4. Restart the server, then re-run a real clip in **real** mode and compare.

**Judge it on video, not only on the validation table.** These numbers are measured
on Open Images photographs; the model's actual failure was on compressed video of a
small, motion-blurred weapon. A higher mAP that does not change what happens on the
ATF and Counter-Strike clips has not fixed the problem that matters.

**If it scores worse, keep run 1.** That is why the old file is preserved.